# Parsing Massey Data

This notebook parses Massey College Football data by conference and year from URLs like this one:  
https://masseyratings.com/scores.php?all=1&format=2&s=587076&sub=10678

That should return the teams in the B1G during 2024; set `format=1` for the games. The code below merges those two data frames into one with the team names and scores. It does so for several years and the power 4/5 conferences. These easiest way to find the query string for conference and year is to navigate from  
https://masseyratings.com/data

Note that the response will be empty, until you provide `all` and `format` values.

In [1]:
import pandas as pd
import time

In [3]:
conferences_by_year = [
    {'conf': 'B1G', 'year': 2025, 'q': 's=627248&sub=10678'},
    {'conf': 'B1G', 'year': 2024, 'q': 's=587076&sub=10678'},
    {'conf': 'ACC', 'year': 2024, 'q': 's=587076&sub=10423'},
    {'conf': 'SEC', 'year': 2024, 'q': 's=587076&sub=14028'},
    {'conf': 'Big12', 'year': 2024, 'q': 's=587076&sub=10686'},

    {'conf': 'ACC', 'year': 2023, 'q': 's=539277&sub=10423'},
    {'conf': 'B1G', 'year': 2023, 'q': 's=539277&sub=10678'},
    {'conf': 'Big12', 'year': 2023, 'q': 's=539277&sub=10686'},
    {'conf': 'SEC', 'year': 2023, 'q': 's=539277&sub=14028'},
    {'conf': 'PAC12', 'year': 2023, 'q': 's=539277&sub=107818'},

    {'conf': 'ACC', 'year': 2022, 'q': 's=384031&sub=10423'},
    {'conf': 'B1G', 'year': 2022, 'q': 's=384031&sub=10678'},
    {'conf': 'Big12', 'year': 2022, 'q': 's=384031&sub=10686'},
    {'conf': 'SEC', 'year': 2022, 'q': 's=384031&sub=14028'},
    {'conf': 'PAC12', 'year': 2022, 'q': 's=384031&sub=107818'},

    {'conf': 'ACC', 'year': 2021, 'q': 's=358435&sub=10423'},
    {'conf': 'B1G', 'year': 2021, 'q': 's=358435&sub=10678'},
    {'conf': 'Big12', 'year': 2021, 'q': 's=358435&sub=10686'},
    {'conf': 'SEC', 'year': 2021, 'q': 's=358435&sub=14028'},
    {'conf': 'PAC12', 'year': 2021, 'q': 's=358435&sub=107818'},

    {'conf': 'ACC', 'year': 2019, 'q': 's=308075&sub=10423'},
    {'conf': 'B1G', 'year': 2019, 'q': 's=308075&sub=10678'},
    {'conf': 'Big12', 'year': 2019, 'q': 's=308075&sub=10686'},
    {'conf': 'SEC', 'year': 2019, 'q': 's=308075&sub=14028'},
    {'conf': 'PAC12', 'year': 2019, 'q': 's=308075&sub=107818'},

    {'conf': 'ACC', 'year': 2018, 'q': 's=300937&sub=10423'},
    {'conf': 'B1G', 'year': 2018, 'q': 's=300937&sub=10678'},
    {'conf': 'Big12', 'year': 2018, 'q': 's=300937&sub=10686'},
    {'conf': 'SEC', 'year': 2018, 'q': 's=300937&sub=14028'},
    {'conf': 'PAC12', 'year': 2018, 'q': 's=300937&sub=107818'},

    {'conf': 'ACC', 'year': 2017, 'q': 's=295489&sub=10423'},
    {'conf': 'B1G', 'year': 2017, 'q': 's=295489&sub=10678'},
    {'conf': 'Big12', 'year': 2017, 'q': 's=295489&sub=10686'},
    {'conf': 'SEC', 'year': 2017, 'q': 's=295489&sub=14028'},
    {'conf': 'PAC12', 'year': 2017, 'q': 's=295489&sub=107818'}
]

In [18]:
len(conferences_by_year)

35

In [4]:
all_games = pd.DataFrame()
for cy in conferences_by_year:
    teams = pd.read_csv(
        f"https://masseyratings.com/scores.php?all=1&format=2&{cy['q']}",
        names = ['massey_id', 'massey_name']
    )
    games = pd.read_csv(
        f"https://masseyratings.com/scores.php?all=1&format=1&{cy['q']}",
        names = ["day","date","team1","home1","score1","team2","home2","score2"],
        parse_dates=['date'], date_format= '%Y%m%d'
    )
    # Add team names and idx (to be used as a matrix index)
    teams['idx'] = teams.index
    reindexed_teams = teams.set_index('massey_id')
    id_to_name_dict = reindexed_teams['massey_name'].to_dict()
    id_to_idx_dict = reindexed_teams['idx'].to_dict()
    games['WTeamName'] = games.team1.apply(lambda t: id_to_name_dict[t].strip())
    games['LTeamName'] = games.team2.apply(lambda t: id_to_name_dict[t].strip())
    # games['team1_idx'] = games.team1.apply(lambda t: id_to_idx_dict[t])
    # games['team2_idx'] = games.team2.apply(lambda t: id_to_idx_dict[t])

    games['WScore'] = games.score1
    games['LScore'] = games.score2
    games['year'] = cy['year']
    games['conf'] = cy['conf']

    all_games = pd.concat(
        [all_games, games[['year', 'conf', 'WTeamName', 'WScore', 'LTeamName', 'LScore']]], 
        ignore_index=True
    )
    time.sleep(1)

In [5]:
all_games

,year,conf,WTeamName,WScore,LTeamName,LScore
0,2025,B1G,Oregon,34,Northwestern,14
1,2025,B1G,USC,33,Purdue,17
2,2025,B1G,Iowa,38,Rutgers,28
3,2025,B1G,Indiana,63,Illinois,10
4,2025,B1G,Michigan,30,Nebraska,27
...,...,...,...,...,...,...
2039,2017,PAC12,Arizona_St,42,Arizona,30
2040,2017,PAC12,Utah,34,Colorado,13
2041,2017,PAC12,Oregon,69,Oregon_St,10
2042,2017,PAC12,Washington,41,Washington_St,14


In [21]:
all_games.to_csv("cfb_games_by_conf_and_year.csv", index=False)

In [23]:
# Read in Massey's list of teams and ids
teams = pd.read_csv(
    'https://masseyratings.com/scores.php?all=1&format=2&s=627248&sub=10678',
    names = ['massey_id', 'massey_name']
)
games = pd.read_csv(
    # 'https://masseyratings.com/scores.php?s=627248&sub=10678&all=1&mode=3&format=1',
    'https://masseyratings.com/scores.php?all=1&format=1&s=627248&sub=10678',
    names = ["day","date","team1","home1","score1","team2","home2","score2"],
    parse_dates=['date'], date_format= '%Y%m%d'
)

# Add team names and idx (to be used as a matrix index)
teams['idx'] = teams.index
reindexed_teams = teams.set_index('massey_id')
id_to_name_dict = reindexed_teams['massey_name'].to_dict()
id_to_idx_dict = reindexed_teams['idx'].to_dict()
games['WTeamName'] = games.team1.apply(lambda t: id_to_name_dict[t].strip())
games['LTeamName'] = games.team2.apply(lambda t: id_to_name_dict[t].strip())
# games['team1_idx'] = games.team1.apply(lambda t: id_to_idx_dict[t])
# games['team2_idx'] = games.team2.apply(lambda t: id_to_idx_dict[t])

games['WScore'] = games.score1
games['LScore'] = games.score2

just_games = games[['WTeamName', 'WScore', 'LTeamName', 'LScore']]
just_games

,WTeamName,WScore,LTeamName,LScore
0,Oregon,34,Northwestern,14
1,USC,33,Purdue,17
2,Iowa,38,Rutgers,28
3,Indiana,63,Illinois,10
4,Michigan,30,Nebraska,27
5,USC,45,Michigan_St,31
6,Maryland,27,Wisconsin,10
7,Illinois,34,USC,32
8,Indiana,20,Iowa,15
9,Minnesota,31,Rutgers,28


In [17]:
games

,day,date,team1,home1,score1,team2,home2,score2
0,739872,2025-09-13,11,-1,34,9,1,14
1,739872,2025-09-13,16,-1,33,13,1,17
2,739878,2025-09-19,3,-1,38,14,1,28
3,739879,2025-09-20,2,1,63,1,-1,10
4,739879,2025-09-20,5,-1,30,8,1,27
5,739879,2025-09-20,16,1,45,6,-1,31
6,739879,2025-09-20,4,-1,27,18,1,10
7,739886,2025-09-27,1,1,34,16,-1,32
8,739886,2025-09-27,2,-1,20,3,1,15
9,739886,2025-09-27,7,1,31,14,-1,28
